In [ ]:
# 1. Load metadata (i: dataset folder directory for LINCS, o: 3 metadata files loaded)
# 2. Inspect metadata (i: 3 metadata files, o: -)
# 3. Analyze Finferprint collision issues => handle issues appropriately
# 4. Load Dataset
# 5. Inspect dataset
# 6. preprocess the data

# Load Packages 

In [ ]:
# Core
import os

import numpy as np
import pandas as pd
from pandarallel import pandarallel  # For making applying of a function faster

pandarallel.initialize(progress_bar=True)

# Show all columns
pd.set_option("display.max_columns", None)
# Disable internal RDKit logs

In [ ]:
# scripts
from data_loading import load_metadata_txt, LINCSDataLoader
from inspect_fingerprints import get_fingerprint, analyze_fingerprint_collision, inspect_reason

In [ ]:
# Utils
from pathlib import Path

# For gctx
from cmapPy.pandasGEXpress.parse import parse

# RDKit stuff
from rdkit import RDLogger

# Calculations

RDLogger.DisableLog("rdApp.*")

In [ ]:
import warnings
from warnings import simplefilter

# Suppress specific FutureWarning from cmapPy/pandas interaction
warnings.filterwarnings("ignore", category=FutureWarning, module="cmapPy")
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

# Load Metadata

In [ ]:
METADATA_EDITED_FOLDER = Path("./Metadata_edited/")
if not os.path.exists(METADATA_EDITED_FOLDER):
    os.mkdir(METADATA_EDITED_FOLDER)

In [ ]:
# Path is user input
merged_LINCS_dataset_dir = Path("/Users/ani/Thesis/Working with LINCS/working_with_lincs/LINCS_datasets/LINCS_metadata")
save_dir_analysis_merged = merged_LINCS_dataset_dir / "Analysis"

In [ ]:
comp_info_merged = load_metadata_txt(merged_LINCS_dataset_dir / "compoundinfo_beta.txt")
gene_info_merged = load_metadata_txt(merged_LINCS_dataset_dir / "geneinfo_beta.txt")
inst_info_merged = load_metadata_txt(merged_LINCS_dataset_dir / "instinfo_beta.txt")

# Inspect Metadata

In [ ]:
inst_info_merged["pert_type"].unique()

In [ ]:
controls = ["ctl_x", "ctl_vehicle", "ctl_untrt", "ctl_vector"] # is hard coded, can be diff from sciplex

inst_info_merged_control = inst_info_merged[
    inst_info_merged["pert_type"].isin(controls)
]

In [ ]:
inst_info_merged_control.head()

In [ ]:
inst_info_merged.head()

In [ ]:
comp_info_merged.head()

In [ ]:
gene_info_merged.head()

In [ ]:
# Subset just for compounds
inst_info_merged_comp = inst_info_merged[inst_info_merged["pert_type"] == "trt_cp"]

In [ ]:
inst_info_merged_comp.head()

# Fingerprint Collisions

In [ ]:
comp_info_merged_path = METADATA_EDITED_FOLDER / "compounds_info_fingerprints.parquet"
if os.path.isfile(comp_info_merged_path):
    # Read Parquet instead of CSV
    comp_info_merged = pd.read_parquet(comp_info_merged_path)
else:
    comp_info_merged["Fingerprint_smiles"] = comp_info_merged[
        "canonical_smiles"
    ].parallel_apply(get_fingerprint)

    # Save to Parquet
    comp_info_merged.to_parquet(comp_info_merged_path, index=False)

In [ ]:
collision_df_path = METADATA_EDITED_FOLDER / "collision_df.parquet"
if os.path.isfile(collision_df_path):
    # Read Parquet instead of CSV
    collision_df_merged = pd.read_parquet(collision_df_path)
else:
    results = []
    for fingerprint, group_df in comp_info_merged.groupby("Fingerprint_smiles"):
        if len(group_df) < 2:
            continue

        smiles_list = group_df["canonical_smiles"].to_list()
        reasons = analyze_fingerprint_collision(smiles_list)

        results.append(
            {
                "Fingerprint": fingerprint,
                "No. compounds": len(smiles_list),
                "Reasons": ", ".join(reasons),
                "Smiles": smiles_list,
                "cmap_name": group_df["cmap_name"].to_list(),
                "pert_id": group_df["pert_id"].to_list(),
            }
        )
    collision_df_merged = pd.DataFrame(results)

    # Save to Parquet
    collision_df_merged.to_parquet(collision_df_path, index=False)

In [ ]:
collision_df_merged.head()

In [ ]:
compounds_per_reasons = dict(
    sorted(
        {
            reas: (group["No. compounds"].sum(), len(group))
            for reas, group in collision_df_merged.groupby("Reasons")
        }.items(),
        key=lambda item: item[1],
        reverse=True,
    )
)
print(
    f"Total number of affected compounds: {np.array(list(compounds_per_reasons.values()))[:, 0].sum()}"
)
print(f"{100 * '-'}\n")
for key, val in compounds_per_reasons.items():
    print(f"{key:<140}: {val[0]:<5} compounds in {val[1]:<5} groups")

In [ ]:
inspect_reason(collision_df_merged, "True duplicates", max_print=20, draw_mols=False)

In [ ]:
inspect_reason(collision_df_merged, "Error in the labelling", max_print=10, draw_mols=True)

In [ ]:
for reason in collision_df_merged["Reasons"].unique().tolist():
    inspect_reason(collision_df_merged, reason, max_print=2, draw_mols=False) # set to true if you want drawings

# Loading GE

In [ ]:
save_dir_analysis = merged_LINCS_dataset_dir / "Analysis"
merged_LINCS_dataset_lvl3_path = (
    merged_LINCS_dataset_dir / "level3_beta_trt_cp_n1805898x12328.gctx"
)

In [ ]:
dataloader = LINCSDataLoader(
    gctx_path=merged_LINCS_dataset_lvl3_path,
    inst_info=inst_info_merged_comp,
    gene_info=gene_info_merged,
    comp_identifier="pert_id",
    cell_identifier="cell_iname",
    instance_identifier="sample_id",
)

In [ ]:
# Assuming dataloader is your instantiated LINCSDataLoader
# (which already filtered for landmark genes via gene_marker="landmark" during init)

inst_filters = {
    # "cell_iname": ["MCF7", "A549"],  # Multiple cell lines
    # "pert_time": 24,  # Single timepoint
    # "pert_dose_unit": "uM",  # Specific dose unit
    "pert_id": "BRD-K79781870"
}
gene_filters = {"feature_space": "landmark"}
filtered_metadata, expression_matrix = dataloader.get_gene_expression(
    inst_filters=inst_filters, gene_filters=gene_filters
)

if expression_matrix is not None:
    print(
        f"Loaded {expression_matrix.shape[0]} profiles and {expression_matrix.shape[1]} genes."
    )

In [ ]:
inst_info_merged_comp[inst_info_merged_comp["pert_id"] == "BRD-K79781870"].shape

In [ ]:
"sample_id" in inst_info_merged_comp.columns

In [ ]:
selectors_inst = [
    "cell_iname",
    "pert_id",
    "nearest_dose",
    "pert_dose_unit",
    "pert_idose",
    "pert_time",
    "pert_itime",
    "pert_time_unit",
]

In [ ]:
gene_info_merged

In [ ]:
selectors_gene = ["gene_symbol", "ensembl_id", "gene_type", "feature_space", "gene_id"]